# CHEM 269 Final Project
## 3D Conformational Descriptors and Dual-Dielectric Solvent Modeling to Decode Cyclic Peptide Membrane Permeation
**Jorge Carmona | March 2026**

---

### Scientific Question
Can 3D descriptors computed across two dielectric environments (aqueous ε=78, membrane-mimetic ε=4) quantify **chameleonic potential** and correlate with experimental membrane permeability (LogPexp) across ~8,000 compounds in **CycPeptMPDB**?

### Pipeline Overview
```
CycPeptMPDB (~8,466 compounds)
       ↓
PAMPA subset (~7,298 with LogPexp)  +  Reference set (CycloA + analogs)
       ↓
Tier-1: RDKit ETKDGv3 conformer ensemble (50 confs/molecule)
        MMFF94s minimization
        Select: aqueous conformer (max-PSA) / membrane conformer (min-PSA)
        Compute: ΔPSA, ΔHB, ΔRg, ΔNPR1, ΔNPR2, PSA-spread
       ↓
Feature matrix: [Tier-1 Δ features] + [DB 3DPSA] + [2D baseline]
       ↓
Analysis: Pearson/Spearman r vs LogPexp | AUC-ROC | UMAP colored by LogPexp
       ↓
Tier-2 cross-check: Tier-1 ΔΨ vs DB CHCl3/H2O 3DPSA (reference set)
```


## 0. Setup

In [ ]:
import sys, os
from pathlib import Path

# ── Colab: mount drive and install env ──────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q rdkit umap-learn hdbscan leidenalg igraph mordred tqdm
    ROOT = Path('/content/drive/MyDrive/CHEM_269_Final_Project')
else:
    ROOT = Path('..').resolve()   # project root when running locally

DATA_CSV   = ROOT / 'CycPeptMPDB_Peptide_All (2).csv'
DATA_DIR   = ROOT / 'data'
RESULTS    = ROOT / 'results'
FIGURES    = RESULTS / 'figures'
SCRIPTS    = ROOT / 'scripts'

for d in [DATA_DIR, RESULTS, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

# Add scripts to path for direct imports
sys.path.insert(0, str(SCRIPTS))

print(f'Root: {ROOT}')
print(f'Data CSV exists: {DATA_CSV.exists()}')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import RobustScaler

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
RDLogger.DisableLog('rdApp.*')

import umap

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
PAMPA_THRESHOLD = -6.0
RANDOM_STATE = 42
CYCLOA_IDS = {1, 22, 932, 981, 1822, 1862, 2356, 7188, 7353}

print('Environment OK')

## 1. Data Loading and Curation

In [ ]:
# Run data curation script
import subprocess
result = subprocess.run(
    [sys.executable, str(SCRIPTS / 'curate_data.py'),
     '--input', str(DATA_CSV), '--outdir', str(DATA_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# Load curated PAMPA dataset
pampa = pd.read_csv(DATA_DIR / 'pampa_curated.csv', low_memory=False)
pampa['permeable'] = (pampa['PAMPA'] >= PAMPA_THRESHOLD).astype(int)

print(f'PAMPA subset: {len(pampa)} compounds')
print(f'  Permeable (>= {PAMPA_THRESHOLD}): {pampa["permeable"].sum()} ({100*pampa["permeable"].mean():.1f}%)')
print(f'  CHCl3/H2O 3DPSA available: {pampa["CHCl3_3DPSA"].notna().sum()}')
print()

# PAMPA distribution plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(pampa['PAMPA'].dropna(), bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax1.axvline(PAMPA_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Threshold = {PAMPA_THRESHOLD}')
ax1.set_xlabel('PAMPA LogPexp (log cm/s)')
ax1.set_ylabel('Count')
ax1.set_title('PAMPA Permeability Distribution (CycPeptMPDB)')
ax1.legend()

# DB 3DPSA: aqueous vs membrane
sub3d = pampa[pampa['CHCl3_3DPSA'].notna() & pampa['H2O_3DPSA'].notna()].copy()
sub3d['delta_3DPSA_db'] = sub3d['H2O_3DPSA'] - sub3d['CHCl3_3DPSA']
ax2.scatter(sub3d['delta_3DPSA_db'], sub3d['PAMPA'],
            s=4, alpha=0.2, c='steelblue', rasterized=True)
# Highlight CycloA
cycloA = sub3d[sub3d['ID'].isin(CYCLOA_IDS)]
ax2.scatter(cycloA['delta_3DPSA_db'], cycloA['PAMPA'],
            s=100, c='black', marker='*', zorder=10, label='CycloA group')
ax2.axhline(PAMPA_THRESHOLD, color='red', linestyle='--', linewidth=1)
ax2.set_xlabel('DB ΔPSA = H₂O_3DPSA − CHCl₃_3DPSA')
ax2.set_ylabel('PAMPA LogPexp (log cm/s)')
ax2.set_title('DB 3D Δ PSA vs. Permeability')
ax2.legend(fontsize=9)
r, p = stats.spearmanr(sub3d['delta_3DPSA_db'].dropna(), sub3d['PAMPA'].dropna())
ax2.text(0.05, 0.95, f'Spearman ρ = {r:.3f}', transform=ax2.transAxes,
         fontsize=9, verticalalignment='top')

plt.tight_layout()
plt.savefig(FIGURES / 'fig1_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {FIGURES}/fig1_data_overview.png')

## 2. Tier-1: Conformer Generation

**Strategy:** ETKDGv3 generates 50 conformers per molecule (macrocycle-aware torsion library). MMFF94s minimization. We then select:
- **Aqueous conformer** = max-PSA conformer (polar groups maximally exposed → water-stable)
- **Membrane conformer** = min-PSA conformer (polar groups buried → membrane-stable)

This approximates dual-dielectric behavior without explicit GB/SA (Tier-1 approximation).

> **Note:** Full run (~7,000 molecules × 50 conformers) takes ~45 min on 4 CPUs. Set `--max-mols 200` for a quick test.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
N_CONFS   = 50     # conformers per molecule
N_CPUS    = 4      # parallel workers
MAX_MOLS  = 0      # 0 = all; set to 200 for quick test

# Check if conformer results already exist
conf_csv = RESULTS / 'conformer_descriptors_raw.csv'
if conf_csv.exists():
    print(f'Conformer results already exist: {conf_csv}')
    print('Delete file and re-run to regenerate.')
else:
    print('Running conformer engine ...')
    cmd = [
        sys.executable, str(SCRIPTS / 'conformer_engine.py'),
        '--input',    str(DATA_DIR / 'pampa_curated.csv'),
        '--outdir',   str(RESULTS),
        '--n-confs',  str(N_CONFS),
        '--n-cpus',   str(N_CPUS),
        '--max-mols', str(MAX_MOLS),
    ]
    import subprocess
    proc = subprocess.run(cmd, capture_output=False)
    print('Done.' if proc.returncode == 0 else f'Error: {proc.returncode}')

In [ ]:
# Inspect conformer results
if conf_csv.exists():
    conf_df = pd.read_csv(conf_csv)
    good = conf_df[conf_df['error'].isna()]
    print(f'Total molecules attempted: {len(conf_df)}')
    print(f'Successful:  {len(good)}')
    print(f'Failed:      {conf_df["error"].notna().sum()}')
    print()
    print('Δ feature stats (successful molecules):')
    delta_cols = [c for c in good.columns if c.startswith('delta_') or c == 'psa3d_spread']
    print(good[delta_cols].describe().round(3).to_string())

## 3. Feature Matrix

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPTS / 'build_feature_matrix.py'),
     '--pampa',      str(DATA_DIR / 'pampa_curated.csv'),
     '--conformers', str(conf_csv),
     '--outdir',     str(RESULTS)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# Load feature matrix
fm = pd.read_csv(RESULTS / 'feature_matrix.csv', low_memory=False)
print(f'Feature matrix: {fm.shape}')
print(f'PAMPA available: {fm["PAMPA"].notna().sum()}')

# Verify key columns
key_cols = ['delta_3DPSA_db', 'delta_psa3d', 'delta_hb', 'delta_Rg',
            'psa3d_spread', 'MolLogP', 'TPSA']
for col in key_cols:
    n = fm[col].notna().sum() if col in fm.columns else 0
    status = '✓' if n > 100 else '✗'
    print(f'  {status} {col:30s}: {n} values')

## 4. Correlation Analysis: Δ Features vs. PAMPA LogPexp

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPTS / 'correlation_analysis.py'),
     '--matrix', str(RESULTS / 'feature_matrix.csv'),
     '--outdir', str(RESULTS)],
    capture_output=True, text=True
)
print(result.stdout[-3000:])  # last 3k chars
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# Display results
corr_df = pd.read_csv(RESULTS / 'correlation_table.csv')
auc_df  = pd.read_csv(RESULTS / 'auc_roc_table.csv')

print('=== Top 15 Features by Spearman ρ ===')
display_cols = ['Feature', 'Group', 'N', 'Pearson_r', 'Spearman_rho', 'Pearson_p']
print(corr_df.head(15)[display_cols].to_string(index=False))

print()
print('=== Top 15 Features by AUC-ROC ===')
print(auc_df.head(15)[['Feature', 'Group', 'N', 'AUC_ROC']].to_string(index=False))

In [ ]:
# Show figures inline
from IPython.display import Image, display
display(Image(str(FIGURES / 'correlation_heatmap.png')))

In [ ]:
display(Image(str(FIGURES / 'auc_roc_bar.png')))

In [ ]:
display(Image(str(FIGURES / 'scatter_top_features.png')))

## 5. UMAP Visualization

### Scaling and dimensionality reduction rationale

Our descriptors span very different numerical ranges:

| Feature | Typical range |
|---------|--------------|
| `MolWt` | 600–1400 Da |
| `MolLogP` | 0–8 (unitless) |
| `TPSA` | 50–300 Å² |
| `delta_psa3d` | 0–100 Å² |
| `delta_hb` | 0–5 (counts) |

Without scaling, large-magnitude features dominate cosine distances by **numerical scale alone**, not chemical relevance.

**Pipeline: RobustScaler → PCA → UMAP (cosine) → Leiden**

- **RobustScaler** (centers to median, scales by IQR): robust to outlier peptides with extreme MW or logP. More appropriate than StandardScaler for this data because cyclic peptide distributions are right-skewed.
- **PCA** on the scaled dense continuous features: the correct dimensionality reduction for this data type. TruncatedSVD is designed for sparse fingerprint matrices and does not center; PCA properly accounts for covariance structure in scaled continuous descriptor space.
- **Silhouette score computed on PCA coordinates** — NOT on the 2D UMAP embedding. UMAP distorts inter-cluster distances in the 2D projection for visualization purposes, so silhouette on the embedding coordinates is meaningless.

In [ ]:
# PCA elbow plot — show how many components are needed per panel
# This validates the PCA step before UMAP
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

fm_plot = pd.read_csv(RESULTS / 'feature_matrix.csv', low_memory=False)

panel_features = {
    'Panel A — 2D descriptors': [
        'MolWt', 'MolLogP', 'TPSA', 'NumHAcceptors',
        'NumHDonors', 'NumRotatableBonds', 'FractionCSP3', 'RingCount',
    ],
    'Panel B — 3D Δ features': [
        'delta_3DPSA_db', 'delta_psa3d', 'delta_hb', 'delta_Rg',
        'delta_NPR1', 'delta_NPR2', 'psa3d_spread', 'psa3d_std',
    ],
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (panel_name, feats) in zip(axes, panel_features.items()):
    avail = [f for f in feats if f in fm_plot.columns]
    sub = fm_plot[avail].dropna()
    if len(sub) < 50 or len(avail) < 3:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        continue

    X = RobustScaler().fit_transform(sub.values)
    pca = PCA(n_components=min(len(avail), len(sub)-1))
    pca.fit(X)

    cum_var = np.cumsum(pca.explained_variance_ratio_) * 100
    ax.bar(range(1, len(cum_var)+1), pca.explained_variance_ratio_*100,
           color='steelblue', alpha=0.7, label='Per-component variance')
    ax.plot(range(1, len(cum_var)+1), cum_var, 'r-o', markersize=4, label='Cumulative variance')
    ax.axhline(90, color='grey', linestyle='--', linewidth=0.8, label='90% threshold')
    ax.set_xlabel('PCA Component')
    ax.set_ylabel('Variance Explained (%)')
    ax.set_title(f'{panel_name}\n(n={len(sub)}, RobustScaler applied first)')
    ax.legend(fontsize=8)
    ax.set_xticks(range(1, len(cum_var)+1))

    n_90 = np.searchsorted(cum_var, 90) + 1
    print(f'{panel_name}: {n_90} components explain 90% variance (of {len(avail)} features)')

plt.suptitle('PCA Elbow Plots — RobustScaler → PCA on Continuous Descriptors', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'pca_elbow_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPTS / 'umap_visualization.py'),
     '--matrix', str(RESULTS / 'feature_matrix.csv'),
     '--outdir', str(RESULTS)],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# UMAP Panel A: 2D descriptors
display(Image(str(FIGURES / 'Panel_A_2D_umap.png')))

In [ ]:
# UMAP Panel B: 3D Δ features
display(Image(str(FIGURES / 'Panel_B_3D_delta_umap.png')))

In [ ]:
# UMAP Panel C: combined
display(Image(str(FIGURES / 'Panel_C_combined_umap.png')))

In [ ]:
# Panel summary
panel_summary = pd.read_csv(RESULTS / 'umap_panel_summary.csv')
print('UMAP Panel Summary:')
print(panel_summary.to_string(index=False))

## 6. Tier-2 Validation: Reference Set Cross-Check

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPTS / 'tier2_validation.py'),
     '--matrix', str(RESULTS / 'feature_matrix.csv'),
     '--refset', str(DATA_DIR / 'reference_set.csv'),
     '--outdir', str(RESULTS)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

In [ ]:
display(Image(str(FIGURES / 'tier2_crosscheck.png')))

## 7. Summary and Conclusions

### Key findings

*(Fill in after running the pipeline)*

1. **DB Δ PSA** (H₂O_3DPSA − CHCl₃_3DPSA): Spearman ρ = ___ vs. PAMPA LogPexp
2. **Tier-1 ΔPSA** (conformer spread): Spearman ρ = ___
3. **AUC-ROC** of best Δ feature: ___
4. **UMAP Panel B** (3D Δ features): silhouette = ___, max cluster enrichment = ___
5. **Tier-2 cross-check**: Tier-1 vs DB r = ___

### Limitations
- RDKit MMFF94s does not explicitly model dielectric environments (Tier-1 approximation)
- Chameleonic conformer selection by PSA extremes is heuristic, not physics-based
- PAMPA assay heterogeneity across sources may add noise
- With n > 6,000 compounds, even weak correlations may be statistically significant — report effect sizes

### Next steps / Thesis connection
- Apply Tier-2 (OMEGA + OpenMM GB/SA) to top 5 reference compounds for direct validation
- Transfer pipeline to in-house DEL library (PI permission required)
- Replace PAMPA threshold binarization with multi-assay consensus label

In [ ]:
# Print full results summary
print('=== RESULTS SUMMARY ===')
print()

if (RESULTS / 'correlation_table.csv').exists():
    corr = pd.read_csv(RESULTS / 'correlation_table.csv')
    top1 = corr.iloc[0]
    print(f'Best feature by Spearman ρ: {top1["Feature"]} (ρ={top1["Spearman_rho"]:.3f}, p={top1["Spearman_p"]:.4f})')

if (RESULTS / 'auc_roc_table.csv').exists():
    auc = pd.read_csv(RESULTS / 'auc_roc_table.csv')
    top_auc = auc.iloc[0]
    print(f'Best feature by AUC-ROC: {top_auc["Feature"]} (AUC={top_auc["AUC_ROC"]:.3f})')

if (RESULTS / 'umap_panel_summary.csv').exists():
    ups = pd.read_csv(RESULTS / 'umap_panel_summary.csv')
    for _, row in ups.iterrows():
        print(f'{row["panel"]}: {row["n_clusters"]} clusters, sil={row["silhouette"]}, max_enrichment={row["max_enrichment"]:.2f}x')

print()
print(f'All figures: {FIGURES}/')
print(f'All results: {RESULTS}/')